In [1]:
import pandas as pd
import numpy as np
import time
import pickle
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, auc, balanced_accuracy_score, 
                             f1_score, recall_score, precision_score)
from sklearn.utils.class_weight import compute_class_weight

# Librerie per resampling
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks, NearMiss, RandomUnderSampler

import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("NOTEBOOK: CONFRONTO APPROCCI - FOCUS CLASSE MINORITARIA")
print("="*80)
print("\n✓ Librerie caricate!\n")

# ==========================================
# SEZIONE 1: CARICAMENTO DATI
# ==========================================
print("="*80)
print("SEZIONE 1: CARICAMENTO DATI")
print("="*80)

CSV_PATH = './BunkerChurners_PearsonCleaned_OutliersRemoved.csv'

print(f"\nCaricamento CSV da: {CSV_PATH}")
df = pd.read_csv(CSV_PATH)

print(f"✓ Dataset caricato: {df.shape}")
print(f"\nPrime righe:")
print(df.head())

print(f"\nDistribuzione target (Attrition_Flag):")
print(df['Attrition_Flag'].value_counts())
print(f"\nRapporto di sbilanciamento: {df['Attrition_Flag'].value_counts().iloc[0] / df['Attrition_Flag'].value_counts().iloc[1]:.2f}:1")


NOTEBOOK: CONFRONTO APPROCCI - FOCUS CLASSE MINORITARIA

✓ Librerie caricate!

SEZIONE 1: CARICAMENTO DATI

Caricamento CSV da: ./BunkerChurners_PearsonCleaned_OutliersRemoved.csv
✓ Dataset caricato: (8183, 16)

Prime righe:
      Attrition_Flag  Dependent_count Education_Level Marital_Status  \
0  Existing Customer                5      Uneducated        Unknown   
1  Existing Customer                2        Graduate        Married   
2  Existing Customer                2        Graduate        Married   
3  Existing Customer                1       Doctorate       Divorced   
4  Attrited Customer                0        Graduate        Married   

  Income_Category Card_Category  Months_on_book  Total_Relationship_Count  \
0         $120K +          Blue              31                         5   
1  Less than $40K          Blue              48                         5   
2         Unknown          Blue              37                         6   
3     $60K - $80K          Blue   

In [2]:
# ==========================================
# SEZIONE 2: SETUP PREPROCESSING
# ==========================================
print("\n" + "="*80)
print("SEZIONE 2: DEFINIZIONE COLONNE E SETUP PREPROCESSING")
print("="*80)

# Definizione colonne
variabili_numeriche = [
    'Dependent_count', 'Months_on_book', 'Total_Relationship_Count',
    'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit',
    'Total_Revolving_Bal', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Ct',
    'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio'
]

education_order = [['Unknown', 'Uneducated', 'High School', 'College', 
                    'Graduate', 'Post-Graduate', 'Doctorate']]

income_order = [['Unknown', 'Less than $40K', '$40K - $60K', '$60K - $80K', 
                 '$80K - $120K', '$120K +']]

categoriche_nominali = ['Card_Category', 'Marital_Status']
categoriche_ordinali_edu = ['Education_Level']
categoriche_ordinali_inc = ['Income_Category']

print(f"\n✓ Variabili numeriche: {len(variabili_numeriche)}")
print(f"✓ Variabili categoriche nominali: {len(categoriche_nominali)}")
print(f"✓ Variabili categoriche ordinali: {len(categoriche_ordinali_edu) + len(categoriche_ordinali_inc)}")

# Transformers
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])

ordinal_edu_transformer = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories=education_order, handle_unknown='use_encoded_value', unknown_value=-1))
])

ordinal_inc_transformer = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories=income_order, handle_unknown='use_encoded_value', unknown_value=-1))
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer completo
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, variabili_numeriche),
        ('ord_edu', ordinal_edu_transformer, categoriche_ordinali_edu),
        ('ord_inc', ordinal_inc_transformer, categoriche_ordinali_inc),
        ('cat', categorical_transformer, categoriche_nominali)
    ],
    remainder='drop'
)

print("\n✓ Preprocessor creato con successo")


SEZIONE 2: DEFINIZIONE COLONNE E SETUP PREPROCESSING

✓ Variabili numeriche: 11
✓ Variabili categoriche nominali: 2
✓ Variabili categoriche ordinali: 2

✓ Preprocessor creato con successo


In [3]:
# ==========================================
# SEZIONE 3: SPLIT TRAIN-TEST
# ==========================================
print("\n" + "="*80)
print("SEZIONE 3: SPLIT TRAIN-TEST E ENCODING TARGET")
print("="*80)

X = df.drop(columns=['Attrition_Flag'])
y = df['Attrition_Flag']

# Encoding target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"\n✓ Target encoding:")
for i, class_name in enumerate(le.classes_):
    print(f"  {class_name} -> {i}")

# Split stratificato
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"\n✓ Split completato:")
print(f"  Train set: {X_train.shape[0]} esempi")
print(f"  Test set: {X_test.shape[0]} esempi")

print(f"\n  Distribuzione train:")
unique_train, counts_train = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique_train, counts_train):
    print(f"    Classe {cls} ({le.classes_[cls]}): {cnt} ({cnt/len(y_train)*100:.2f}%)")

print(f"\n  Distribuzione test:")
unique_test, counts_test = np.unique(y_test, return_counts=True)
for cls, cnt in zip(unique_test, counts_test):
    print(f"    Classe {cls} ({le.classes_[cls]}): {cnt} ({cnt/len(y_test)*100:.2f}%)")


SEZIONE 3: SPLIT TRAIN-TEST E ENCODING TARGET

✓ Target encoding:
  Attrited Customer -> 0
  Existing Customer -> 1

✓ Split completato:
  Train set: 6546 esempi
  Test set: 1637 esempi

  Distribuzione train:
    Classe 0 (Attrited Customer): 1061 (16.21%)
    Classe 1 (Existing Customer): 5485 (83.79%)

  Distribuzione test:
    Classe 0 (Attrited Customer): 265 (16.19%)
    Classe 1 (Existing Customer): 1372 (83.81%)


In [4]:
# ==========================================
# FUNZIONE HELPER: VALUTAZIONE MODELLI
# ==========================================

def evaluate_model(y_pred, y_pred_proba, y_test_actual, model_name, le_classes):
    """
    Valuta un modello con metriche complete per ENTRAMBE le classi
    
    FOCUS: Classe 0 (Attrited Customer) - quella che ci interessa!
    """
    print(f"\n{'='*80}")
    print(f"RISULTATI: {model_name}")
    print(f"{'='*80}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_test_actual, y_pred)
    print(f"\nConfusion Matrix:")
    print(f"                  Predicted 0    Predicted 1")
    print(f"Actual 0          {cm[0,0]:6d}         {cm[0,1]:6d}")
    print(f"Actual 1          {cm[1,0]:6d}         {cm[1,1]:6d}")
    
    print(f"\nTrue Negatives (TN):  {cm[0,0]}")
    print(f"False Positives (FP): {cm[0,1]}")
    print(f"False Negatives (FN): {cm[1,0]}")
    print(f"True Positives (TP):  {cm[1,1]}")
    
    # Classification Report
    print(f"\nClassification Report:")
    print(classification_report(y_test_actual, y_pred, target_names=le_classes))
    
    # Metriche per CLASSE 0 (MINORITARIA - quella che ci interessa!)
    precision_0 = precision_score(y_test_actual, y_pred, pos_label=0)
    recall_0 = recall_score(y_test_actual, y_pred, pos_label=0)
    f1_0 = f1_score(y_test_actual, y_pred, pos_label=0)
    
    # Metriche per CLASSE 1 (MAGGIORITARIA)
    precision_1 = precision_score(y_test_actual, y_pred, pos_label=1)
    recall_1 = recall_score(y_test_actual, y_pred, pos_label=1)
    f1_1 = f1_score(y_test_actual, y_pred, pos_label=1)
    
    # Metriche globali
    balanced_acc = balanced_accuracy_score(y_test_actual, y_pred)
    roc_auc = roc_auc_score(y_test_actual, y_pred_proba)
    
    precision_vals, recall_vals, _ = precision_recall_curve(y_test_actual, y_pred_proba)
    pr_auc = auc(recall_vals, precision_vals)
    
    print(f"{'='*80}")
    print(f"⭐ METRICHE CHIAVE - CLASSE 0: {le_classes[0]} (PRIORITARIA!)")
    print(f"{'='*80}")
    print(f"Precision (Classe 0): {precision_0:.4f}")
    print(f"Recall (Classe 0):    {recall_0:.4f}  ← METRICA PRINCIPALE!")
    print(f"F1-Score (Classe 0):  {f1_0:.4f}")
    
    print(f"\n{'='*80}")
    print(f"METRICHE CHIAVE - CLASSE 1: {le_classes[1]} (Maggioritaria)")
    print(f"{'='*80}")
    print(f"Precision (Classe 1): {precision_1:.4f}")
    print(f"Recall (Classe 1):    {recall_1:.4f}")
    print(f"F1-Score (Classe 1):  {f1_1:.4f}")
    
    print(f"\n{'='*80}")
    print(f"METRICHE GLOBALI")
    print(f"{'='*80}")
    print(f"Balanced Accuracy:    {balanced_acc:.4f}")
    print(f"ROC-AUC:              {roc_auc:.4f}")
    print(f"PR-AUC:               {pr_auc:.4f}")
    
    return {
        'model_name': model_name,
        # Classe 0 (minoritaria)
        'precision_0': precision_0,
        'recall_0': recall_0,
        'f1_0': f1_0,
        # Classe 1 (maggioritaria)
        'precision_1': precision_1,
        'recall_1': recall_1,
        'f1_1': f1_1,
        # Globali
        'balanced_acc': balanced_acc,
        'roc_auc': roc_auc,
        'pr_auc': pr_auc
    }


In [5]:
# ==========================================
# APPROCCIO 0: BASELINE
# ==========================================
print("\n\n" + "="*80)
print("APPROCCIO 0: BASELINE - SOLO CLASS WEIGHTING")
print("="*80)
print("\nDescrizione:")
print("  - Usa tutti i dati di training senza resampling")
print("  - Applica class_weight='balanced' in Random Forest")
print("  - Preprocessing integrato nella pipeline")
print("  - Questo è il modello di riferimento")

class_weights_baseline = compute_class_weight(
    class_weight='balanced', classes=np.unique(y_train), y=y_train
)
class_weight_dict_baseline = {i: w for i, w in enumerate(class_weights_baseline)}

print(f"\nClass weights calcolati:")
for cls, weight in class_weight_dict_baseline.items():
    print(f"  Classe {cls} ({le.classes_[cls]}): {weight:.4f}")

print(f"\nDimensione training set: {X_train.shape[0]} esempi")

print("\nCreazione pipeline completa e training...")
start_time = time.time()

baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        random_state=42, n_estimators=400, max_depth=15,
        min_samples_split=10, min_samples_leaf=4,
        class_weight=class_weight_dict_baseline, n_jobs=-1
    ))
])

baseline_pipeline.fit(X_train, y_train)
baseline_time = time.time() - start_time

print(f"✓ Training completato in {baseline_time:.2f} secondi")

y_pred_baseline = baseline_pipeline.predict(X_test)
y_pred_proba_baseline = baseline_pipeline.predict_proba(X_test)[:, 1]

results_baseline = evaluate_model(
    y_pred_baseline, y_pred_proba_baseline, y_test,
    "BASELINE (Class Weighting)", le.classes_
)



APPROCCIO 0: BASELINE - SOLO CLASS WEIGHTING

Descrizione:
  - Usa tutti i dati di training senza resampling
  - Applica class_weight='balanced' in Random Forest
  - Preprocessing integrato nella pipeline
  - Questo è il modello di riferimento

Class weights calcolati:
  Classe 0 (Attrited Customer): 3.0848
  Classe 1 (Existing Customer): 0.5967

Dimensione training set: 6546 esempi

Creazione pipeline completa e training...
✓ Training completato in 0.55 secondi

RISULTATI: BASELINE (Class Weighting)

Confusion Matrix:
                  Predicted 0    Predicted 1
Actual 0             198             67
Actual 1              78           1294

True Negatives (TN):  198
False Positives (FP): 67
False Negatives (FN): 78
True Positives (TP):  1294

Classification Report:
                   precision    recall  f1-score   support

Attrited Customer       0.72      0.75      0.73       265
Existing Customer       0.95      0.94      0.95      1372

         accuracy                        

In [6]:
# ==========================================
# PREPROCESSING PER APPROCCI DI RESAMPLING
# ==========================================
print("\n" + "="*80)
print("PREPROCESSING PER APPROCCI DI RESAMPLING")
print("="*80)
print("\nNOTA: Preprocessing separato per evitare data leakage")
print("      Il preprocessor viene fittato UNA VOLTA su train, poi riusato\n")

preprocessor_for_resampling = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, variabili_numeriche),
        ('ord_edu', ordinal_edu_transformer, categoriche_ordinali_edu),
        ('ord_inc', ordinal_inc_transformer, categoriche_ordinali_inc),
        ('cat', categorical_transformer, categoriche_nominali)
    ],
    remainder='drop'
)

X_train_processed = preprocessor_for_resampling.fit_transform(X_train)
X_test_processed = preprocessor_for_resampling.transform(X_test)

print(f"✓ Training set preprocessato: {X_train_processed.shape}")
print(f"✓ Test set preprocessato: {X_test_processed.shape}")


PREPROCESSING PER APPROCCI DI RESAMPLING

NOTA: Preprocessing separato per evitare data leakage
      Il preprocessor viene fittato UNA VOLTA su train, poi riusato

✓ Training set preprocessato: (6546, 19)
✓ Test set preprocessato: (1637, 19)


In [7]:
# ==========================================
# APPROCCIO 1: TOMEK LINKS
# ==========================================
print("\n\n" + "="*80)
print("APPROCCIO 1: TOMEK LINKS")
print("="*80)
print("\nDescrizione:")
print("  - Rimuove esempi della classe maggioritaria vicini alla minoritaria")
print("  - Pulisce il boundary decisionale")
print("  - Preserva TUTTI gli esempi della classe minoritaria")

print(f"\nDimensione originale: {X_train_processed.shape[0]} esempi")

print("Applicazione Tomek Links...")
start_resample = time.time()
tomek = TomekLinks(sampling_strategy='majority')
X_train_tomek, y_train_tomek = tomek.fit_resample(X_train_processed, y_train)
resample_time = time.time() - start_resample

print(f"✓ Resampling completato in {resample_time:.2f} secondi")
print(f"\nDimensione dopo Tomek Links: {X_train_tomek.shape[0]} esempi")
print(f"Esempi rimossi: {X_train_processed.shape[0] - X_train_tomek.shape[0]}")

print(f"\nDistribuzione classi dopo Tomek Links:")
unique, counts = np.unique(y_train_tomek, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Classe {cls} ({le.classes_[cls]}): {cnt} ({cnt/len(y_train_tomek)*100:.2f}%)")

class_weights_tomek = compute_class_weight(
    class_weight='balanced', classes=np.unique(y_train_tomek), y=y_train_tomek
)
class_weight_dict_tomek = {i: w for i, w in enumerate(class_weights_tomek)}

print("\nTraining modello...")
start_time = time.time()

model_tomek = RandomForestClassifier(
    random_state=42, n_estimators=400, max_depth=15,
    min_samples_split=10, min_samples_leaf=4,
    class_weight=class_weight_dict_tomek, n_jobs=-1
)

model_tomek.fit(X_train_tomek, y_train_tomek)
tomek_time = time.time() - start_time

print(f"✓ Training completato in {tomek_time:.2f} secondi")

y_pred_tomek = model_tomek.predict(X_test_processed)
y_pred_proba_tomek = model_tomek.predict_proba(X_test_processed)[:, 1]

results_tomek = evaluate_model(
    y_pred_tomek, y_pred_proba_tomek, y_test,
    "Tomek Links", le.classes_
)



APPROCCIO 1: TOMEK LINKS

Descrizione:
  - Rimuove esempi della classe maggioritaria vicini alla minoritaria
  - Pulisce il boundary decisionale
  - Preserva TUTTI gli esempi della classe minoritaria

Dimensione originale: 6546 esempi
Applicazione Tomek Links...
✓ Resampling completato in 0.16 secondi

Dimensione dopo Tomek Links: 6406 esempi
Esempi rimossi: 140

Distribuzione classi dopo Tomek Links:
  Classe 0 (Attrited Customer): 1061 (16.56%)
  Classe 1 (Existing Customer): 5345 (83.44%)

Training modello...
✓ Training completato in 0.67 secondi

RISULTATI: Tomek Links

Confusion Matrix:
                  Predicted 0    Predicted 1
Actual 0             202             63
Actual 1              82           1290

True Negatives (TN):  202
False Positives (FP): 63
False Negatives (FN): 82
True Positives (TP):  1290

Classification Report:
                   precision    recall  f1-score   support

Attrited Customer       0.71      0.76      0.74       265
Existing Customer       0.9

In [8]:
# ==========================================
# APPROCCIO 2: NEARMISS
# ==========================================
print("\n\n" + "="*80)
print("APPROCCIO 2: NEARMISS")
print("="*80)
print("\nDescrizione:")
print("  - Seleziona esempi difficili della classe maggioritaria")
print("  - Mantiene casi vicini al boundary")
print("  - Riduce significativamente il dataset maggioritario")

print(f"\nDimensione originale: {X_train_processed.shape[0]} esempi")

print("Applicazione NearMiss (version 2)...")
start_resample = time.time()
nearmiss = NearMiss(version=2, n_jobs=-1)
X_train_nm, y_train_nm = nearmiss.fit_resample(X_train_processed, y_train)
resample_time = time.time() - start_resample

print(f"✓ Resampling completato in {resample_time:.2f} secondi")
print(f"\nDimensione dopo NearMiss: {X_train_nm.shape[0]} esempi")
print(f"Esempi rimossi: {X_train_processed.shape[0] - X_train_nm.shape[0]}")

print(f"\nDistribuzione classi dopo NearMiss:")
unique, counts = np.unique(y_train_nm, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Classe {cls} ({le.classes_[cls]}): {cnt} ({cnt/len(y_train_nm)*100:.2f}%)")

class_weights_nm = compute_class_weight(
    class_weight='balanced', classes=np.unique(y_train_nm), y=y_train_nm
)
class_weight_dict_nm = {i: w for i, w in enumerate(class_weights_nm)}

print("\nTraining modello...")
start_time = time.time()

model_nm = RandomForestClassifier(
    random_state=42, n_estimators=400, max_depth=15,
    min_samples_split=10, min_samples_leaf=4,
    class_weight=class_weight_dict_nm, n_jobs=-1
)

model_nm.fit(X_train_nm, y_train_nm)
nm_time = time.time() - start_time

print(f"✓ Training completato in {nm_time:.2f} secondi")

y_pred_nm = model_nm.predict(X_test_processed)
y_pred_proba_nm = model_nm.predict_proba(X_test_processed)[:, 1]

results_nm = evaluate_model(
    y_pred_nm, y_pred_proba_nm, y_test,
    "NearMiss", le.classes_
)




APPROCCIO 2: NEARMISS

Descrizione:
  - Seleziona esempi difficili della classe maggioritaria
  - Mantiene casi vicini al boundary
  - Riduce significativamente il dataset maggioritario

Dimensione originale: 6546 esempi
Applicazione NearMiss (version 2)...
✓ Resampling completato in 0.11 secondi

Dimensione dopo NearMiss: 2122 esempi
Esempi rimossi: 4424

Distribuzione classi dopo NearMiss:
  Classe 0 (Attrited Customer): 1061 (50.00%)
  Classe 1 (Existing Customer): 1061 (50.00%)

Training modello...
✓ Training completato in 0.32 secondi

RISULTATI: NearMiss

Confusion Matrix:
                  Predicted 0    Predicted 1
Actual 0             238             27
Actual 1             330           1042

True Negatives (TN):  238
False Positives (FP): 27
False Negatives (FN): 330
True Positives (TP):  1042

Classification Report:
                   precision    recall  f1-score   support

Attrited Customer       0.42      0.90      0.57       265
Existing Customer       0.97      0.76 

In [9]:
# ==========================================
# APPROCCIO 3: RANDOM UNDERSAMPLING + ENSEMBLE
# ==========================================
print("\n\n" + "="*80)
print("APPROCCIO 3: RANDOM UNDERSAMPLING + ENSEMBLE")
print("="*80)
print("\nDescrizione:")
print("  - Crea 5 modelli con campionamenti casuali diversi")
print("  - Ogni modello usa TUTTI i minoritari + subset maggioritari")
print("  - Predizione finale: soft voting")
print("  - Obiettivo: usare tutti i dati senza perdere informazione")

N_MODELS = 5
n_majority = (y_train == 1).sum()
n_minority = (y_train == 0).sum()
sample_size_majority = n_majority // 2

print(f"\nConfigurazione ensemble:")
print(f"  Numero di modelli: {N_MODELS}")
print(f"  Esempi minoritari per modello: {n_minority} (tutti)")
print(f"  Esempi maggioritari per modello: {sample_size_majority} (~50%)")

ensemble_models = []
total_ensemble_time = 0

print(f"\nTraining {N_MODELS} modelli...")

for i in range(N_MODELS):
    print(f"\n  Modello {i+1}/{N_MODELS}:")
    
    rus = RandomUnderSampler(
        sampling_strategy={1: sample_size_majority, 0: n_minority},
        random_state=42 + i
    )
    X_train_rus, y_train_rus = rus.fit_resample(X_train_processed, y_train)
    
    print(f"    Resampling: {X_train_rus.shape[0]} esempi")
    
    class_weights_rus = compute_class_weight(
        class_weight='balanced', classes=np.unique(y_train_rus), y=y_train_rus
    )
    class_weight_dict_rus = {j: w for j, w in enumerate(class_weights_rus)}
    
    start_time = time.time()
    model_rus = RandomForestClassifier(
        random_state=42 + i, n_estimators=400, max_depth=15,
        min_samples_split=10, min_samples_leaf=4,
        class_weight=class_weight_dict_rus, n_jobs=-1
    )
    model_rus.fit(X_train_rus, y_train_rus)
    elapsed = time.time() - start_time
    total_ensemble_time += elapsed
    
    print(f"    Training: {elapsed:.2f}s")
    
    ensemble_models.append(model_rus)

print(f"\n✓ Ensemble completato in {total_ensemble_time:.2f} secondi totali")

print("\nCalcolo predizioni ensemble (soft voting)...")
ensemble_proba = np.zeros((len(X_test_processed), 2))

for model in ensemble_models:
    ensemble_proba += model.predict_proba(X_test_processed)

ensemble_proba /= N_MODELS
ensemble_pred = np.argmax(ensemble_proba, axis=1)

results_ensemble = evaluate_model(
    ensemble_pred, ensemble_proba[:, 1], y_test,
    "Random Undersampling + Ensemble", le.classes_
)



APPROCCIO 3: RANDOM UNDERSAMPLING + ENSEMBLE

Descrizione:
  - Crea 5 modelli con campionamenti casuali diversi
  - Ogni modello usa TUTTI i minoritari + subset maggioritari
  - Predizione finale: soft voting
  - Obiettivo: usare tutti i dati senza perdere informazione

Configurazione ensemble:
  Numero di modelli: 5
  Esempi minoritari per modello: 1061 (tutti)
  Esempi maggioritari per modello: 2742 (~50%)

Training 5 modelli...

  Modello 1/5:
    Resampling: 3803 esempi
    Training: 0.40s

  Modello 2/5:
    Resampling: 3803 esempi
    Training: 0.37s

  Modello 3/5:
    Resampling: 3803 esempi
    Training: 0.37s

  Modello 4/5:
    Resampling: 3803 esempi
    Training: 0.37s

  Modello 5/5:
    Resampling: 3803 esempi
    Training: 0.37s

✓ Ensemble completato in 1.89 secondi totali

Calcolo predizioni ensemble (soft voting)...

RISULTATI: Random Undersampling + Ensemble

Confusion Matrix:
                  Predicted 0    Predicted 1
Actual 0             212             53
Act

In [10]:
# ==========================================
# APPROCCIO 4: SMOTE + UNDERSAMPLING
# ==========================================
print("\n\n" + "="*80)
print("APPROCCIO 4: SMOTE + UNDERSAMPLING (HYBRID)")
print("="*80)
print("\nDescrizione:")
print("  - SMOTE: genera esempi sintetici della classe minoritaria")
print("  - Undersampling: riduce la classe maggioritaria")
print("  - Bilanciamento moderato (rapporto 2:1)")

TARGET_MINORITY = 2500
TARGET_MAJORITY = 5000

print(f"\nTarget configurazione:")
print(f"  Classe minoritaria (con SMOTE): {TARGET_MINORITY}")
print(f"  Classe maggioritaria (undersampled): {TARGET_MAJORITY}")
print(f"  Rapporto finale: {TARGET_MAJORITY/TARGET_MINORITY:.1f}:1")

print(f"\nStep 1: Applicazione SMOTE...")
start_resample = time.time()
smote = SMOTE(sampling_strategy={0: TARGET_MINORITY}, random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train)
smote_time = time.time() - start_resample

print(f"✓ SMOTE completato in {smote_time:.2f} secondi")

print(f"\nStep 2: Applicazione Random Undersampling...")
start_resample = time.time()
rus = RandomUnderSampler(
    sampling_strategy={1: TARGET_MAJORITY, 0: TARGET_MINORITY},
    random_state=42
)
X_train_hybrid, y_train_hybrid = rus.fit_resample(X_train_smote, y_train_smote)
rus_time = time.time() - start_resample

print(f"✓ Undersampling completato in {rus_time:.2f} secondi")
print(f"\nDataset finale: {len(y_train_hybrid)} esempi")

class_weights_hybrid = compute_class_weight(
    class_weight='balanced', classes=np.unique(y_train_hybrid), y=y_train_hybrid
)
class_weight_dict_hybrid = {i: w for i, w in enumerate(class_weights_hybrid)}

print("\nTraining modello...")
start_time = time.time()

model_hybrid = RandomForestClassifier(
    random_state=42, n_estimators=400, max_depth=15,
    min_samples_split=10, min_samples_leaf=4,
    class_weight=class_weight_dict_hybrid, n_jobs=-1
)

model_hybrid.fit(X_train_hybrid, y_train_hybrid)
hybrid_time = time.time() - start_time

print(f"✓ Training completato in {hybrid_time:.2f} secondi")

y_pred_hybrid = model_hybrid.predict(X_test_processed)
y_pred_proba_hybrid = model_hybrid.predict_proba(X_test_processed)[:, 1]

results_hybrid = evaluate_model(
    y_pred_hybrid, y_pred_proba_hybrid, y_test,
    "SMOTE + Undersampling", le.classes_
)




APPROCCIO 4: SMOTE + UNDERSAMPLING (HYBRID)

Descrizione:
  - SMOTE: genera esempi sintetici della classe minoritaria
  - Undersampling: riduce la classe maggioritaria
  - Bilanciamento moderato (rapporto 2:1)

Target configurazione:
  Classe minoritaria (con SMOTE): 2500
  Classe maggioritaria (undersampled): 5000
  Rapporto finale: 2.0:1

Step 1: Applicazione SMOTE...
✓ SMOTE completato in 0.02 secondi

Step 2: Applicazione Random Undersampling...
✓ Undersampling completato in 0.00 secondi

Dataset finale: 7500 esempi

Training modello...
✓ Training completato in 0.58 secondi

RISULTATI: SMOTE + Undersampling

Confusion Matrix:
                  Predicted 0    Predicted 1
Actual 0             200             65
Actual 1              79           1293

True Negatives (TN):  200
False Positives (FP): 65
False Negatives (FN): 79
True Positives (TP):  1293

Classification Report:
                   precision    recall  f1-score   support

Attrited Customer       0.72      0.75      0.7

In [11]:
# ==========================================
# CONFRONTO FINALE - ENTRAMBE LE CLASSI
# ==========================================
print("\n\n" + "="*80)
print("CONFRONTO FINALE TRA TUTTI GLI APPROCCI")
print("="*80)

results_df = pd.DataFrame([
    results_baseline, results_tomek, results_nm,
    results_ensemble, results_hybrid
])

# CLASSE 0 (MINORITARIA - PRIORITARIA!)
print("\n" + "─"*80)
print("⭐ TABELLA COMPARATIVA - CLASSE 0 (ATTRITED CUSTOMER) - PRIORITARIA!")
print("─"*80)
print(f"\n{'Modello':<35} {'Prec':>7} {'Recall':>7} {'F1':>7}")
print("─"*80)

for _, row in results_df.iterrows():
    print(f"{row['model_name']:<35} "
          f"{row['precision_0']:>7.4f} "
          f"{row['recall_0']:>7.4f} "
          f"{row['f1_0']:>7.4f}")
print("─"*80)

# CLASSE 1 (MAGGIORITARIA)
print("\n" + "─"*80)
print("TABELLA COMPARATIVA - CLASSE 1 (EXISTING CUSTOMER) - Maggioritaria")
print("─"*80)
print(f"\n{'Modello':<35} {'Prec':>7} {'Recall':>7} {'F1':>7}")
print("─"*80)

for _, row in results_df.iterrows():
    print(f"{row['model_name']:<35} "
          f"{row['precision_1']:>7.4f} "
          f"{row['recall_1']:>7.4f} "
          f"{row['f1_1']:>7.4f}")
print("─"*80)

# METRICHE GLOBALI
print("\n" + "─"*80)
print("TABELLA COMPARATIVA - METRICHE GLOBALI")
print("─"*80)
print(f"\n{'Modello':<35} {'Bal.Acc':>8} {'ROC-AUC':>8} {'PR-AUC':>8}")
print("─"*80)

for _, row in results_df.iterrows():
    print(f"{row['model_name']:<35} "
          f"{row['balanced_acc']:>8.4f} "
          f"{row['roc_auc']:>8.4f} "
          f"{row['pr_auc']:>8.4f}")
print("─"*80)



CONFRONTO FINALE TRA TUTTI GLI APPROCCI

────────────────────────────────────────────────────────────────────────────────
⭐ TABELLA COMPARATIVA - CLASSE 0 (ATTRITED CUSTOMER) - PRIORITARIA!
────────────────────────────────────────────────────────────────────────────────

Modello                                Prec  Recall      F1
────────────────────────────────────────────────────────────────────────────────
BASELINE (Class Weighting)           0.7174  0.7472  0.7320
Tomek Links                          0.7113  0.7623  0.7359
NearMiss                             0.4190  0.8981  0.5714
Random Undersampling + Ensemble      0.6861  0.8000  0.7387
SMOTE + Undersampling                0.7168  0.7547  0.7353
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
TABELLA COMPARATIVA - CLASSE 1 (EXISTING CUSTOMER) - Maggioritaria
───────────────────────────────────────────────────────

In [12]:
# ==========================================
# SCORE AGGREGATO PESATO
# ==========================================
print("\n\n" + "="*80)
print("SCORE AGGREGATO PESATO - FOCUS CLASSE MINORITARIA")
print("="*80)
print("\nFormula: 0.60 * Recall_0 + 0.20 * Precision_0 + 0.10 * F1_0 + 0.10 * ROC-AUC")
print("Priorità: RECALL della classe minoritaria (60%)\n")

results_df['weighted_score'] = (
    0.60 * results_df['recall_0'] +
    0.20 * results_df['precision_0'] +
    0.10 * results_df['f1_0'] +
    0.10 * results_df['roc_auc']
)

results_df_sorted = results_df.sort_values('weighted_score', ascending=False)

print(f"{'Modello':<35} {'Score':>8}")
print("─"*80)
for _, row in results_df_sorted.iterrows():
    print(f"{row['model_name']:<35} {row['weighted_score']:>8.4f}")
print("─"*80)



SCORE AGGREGATO PESATO - FOCUS CLASSE MINORITARIA

Formula: 0.60 * Recall_0 + 0.20 * Precision_0 + 0.10 * F1_0 + 0.10 * ROC-AUC
Priorità: RECALL della classe minoritaria (60%)

Modello                                Score
────────────────────────────────────────────────────────────────────────────────
Random Undersampling + Ensemble       0.7866
NearMiss                              0.7724
Tomek Links                           0.7687
SMOTE + Undersampling                 0.7653
BASELINE (Class Weighting)            0.7606
────────────────────────────────────────────────────────────────────────────────


In [13]:
# ==========================================
# RACCOMANDAZIONE FINALE
# ==========================================
print("\n\n" + "="*80)
print("🎯 RACCOMANDAZIONE FINALE")
print("="*80)

best_model = results_df_sorted.iloc[0]
print(f"\n✓ Modello migliore: {best_model['model_name']}")
print(f"  Score aggregato: {best_model['weighted_score']:.4f}")
print(f"\n  Metriche Classe 0 (Attrited Customer):")
print(f"    - Recall:    {best_model['recall_0']:.4f} ⭐")
print(f"    - Precision: {best_model['precision_0']:.4f}")
print(f"    - F1-Score:  {best_model['f1_0']:.4f}")
print(f"\n  Metriche Classe 1 (Existing Customer):")
print(f"    - Recall:    {best_model['recall_1']:.4f}")
print(f"    - Precision: {best_model['precision_1']:.4f}")
print(f"    - F1-Score:  {best_model['f1_1']:.4f}")
print(f"\n  Metriche Globali:")
print(f"    - Balanced Accuracy: {best_model['balanced_acc']:.4f}")
print(f"    - ROC-AUC:           {best_model['roc_auc']:.4f}")
print(f"    - PR-AUC:            {best_model['pr_auc']:.4f}")

print("\n" + "="*80)
print("INTERPRETAZIONE E RACCOMANDAZIONI")
print("="*80)

print("\n📊 ANALISI COMPARATIVA:")
print("\n1. BASELINE (Class Weighting):")
print("   + Semplice e veloce")
print("   + Usa tutti i dati")
print("   - Potrebbe non massimizzare recall su classe minoritaria")

print("\n2. TOMEK LINKS:")
print("   + Pulisce boundary decisionale")
print("   + Preserva TUTTI gli esempi minoritari")
print("   - Rimozione minima, impatto limitato")

print("\n3. NEARMISS:")
print("   + Seleziona esempi difficili")
print("   + Bilanciamento forte")
print("   - Può perdere informazioni importanti dalla maggioritaria")

print("\n4. RANDOM UNDERSAMPLING + ENSEMBLE:")
print("   + Usa tutti i dati attraverso l'ensemble")
print("   + Riduce il rischio di perdita informazioni")
print("   - Più lento da trainare")

print("\n5. SMOTE + UNDERSAMPLING:")
print("   + Bilancia oversampling e undersampling")
print("   + Genera dati sintetici per la minoritaria")
print("   - Rischio overfitting su dati sintetici")

print("\n" + "="*80)
print("💡 CONSIDERAZIONI BUSINESS")
print("="*80)
print("\nPer identificare CHURNERS (Classe 0 - Attrited Customer):")
print("  ✓ RECALL è la metrica più importante")
print("  ✓ Vogliamo catturare il maggior numero possibile di churner")
print("  ✓ I falsi positivi sono meno costosi dei falsi negativi")
print("  ✓ Meglio contattare clienti a rischio anche se stabili,")
print("    piuttosto che perdere churner non identificati")



🎯 RACCOMANDAZIONE FINALE

✓ Modello migliore: Random Undersampling + Ensemble
  Score aggregato: 0.7866

  Metriche Classe 0 (Attrited Customer):
    - Recall:    0.8000 ⭐
    - Precision: 0.6861
    - F1-Score:  0.7387

  Metriche Classe 1 (Existing Customer):
    - Recall:    0.9293
    - Precision: 0.9601
    - F1-Score:  0.9444

  Metriche Globali:
    - Balanced Accuracy: 0.8647
    - ROC-AUC:           0.9550
    - PR-AUC:            0.9906

INTERPRETAZIONE E RACCOMANDAZIONI

📊 ANALISI COMPARATIVA:

1. BASELINE (Class Weighting):
   + Semplice e veloce
   + Usa tutti i dati
   - Potrebbe non massimizzare recall su classe minoritaria

2. TOMEK LINKS:
   + Pulisce boundary decisionale
   + Preserva TUTTI gli esempi minoritari
   - Rimozione minima, impatto limitato

3. NEARMISS:
   + Seleziona esempi difficili
   + Bilanciamento forte
   - Può perdere informazioni importanti dalla maggioritaria

4. RANDOM UNDERSAMPLING + ENSEMBLE:
   + Usa tutti i dati attraverso l'ensemble
   + 

# DashBoard di comparazione per decisione del miglior modello

In [16]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from sklearn.metrics import confusion_matrix
import pickle

"""
🎨 DASHBOARD INTERATTIVA - CONFRONTO APPROCCI RESAMPLING
=========================================================

Questo script crea una dashboard HTML interattiva che visualizza:
1. Matrici di confusione per ogni modello (stesso schema colori)
2. Metriche TN, FP, FN, TP sotto ogni matrice
3. Grafici comparativi per classe minoritaria
4. Grafici comparativi per classe maggioritaria  
5. Metriche globali a confronto
6. Score aggregato finale

PREREQUISITI:
- File 'comparison_results.csv' generato dal notebook principale
- File 'model_metadata.pkl' per le predizioni
"""

# ==========================================
# CONFIGURAZIONE E CARICAMENTO DATI
# ==========================================

print("="*80)
print("📊 CREAZIONE DASHBOARD INTERATTIVA")
print("="*80)

# Colori consistenti per le classi
COLOR_CLASS_0 = '#FF6B6B'  # Rosso per classe minoritaria (Churners)
COLOR_CLASS_1 = '#4ECDC4'  # Turchese per classe maggioritaria (Existing)
COLOR_NEUTRAL = '#95A5A6'  # Grigio per valori neutri

# Carica risultati
print("\n📂 Caricamento dati...")
results_df = pd.read_csv('comparison_results.csv')
print(f"✓ Caricati risultati per {len(results_df)} modelli")

# Simula le confusion matrices (normalmente verrebbero dalle predizioni salvate)
# In un caso reale, ricaricheresti i modelli e ricalcoleresti le CM
print("📊 Generazione matrici di confusione...")

# Per questo esempio, generiamo CM simulate basate sulle metriche
# In produzione, useresti le predizioni reali salvate
confusion_matrices = {}

for _, row in results_df.iterrows():
    model_name = row['model_name']
    
    # Parametri approssimativi per generare CM coerenti con le metriche
    # (In realtà useresti y_test e y_pred salvati)
    total_samples = 1637  # Esempio: dimensione test set
    n_class_0 = 265      # Esempio: campioni classe 0 nel test
    n_class_1 = 1372     # Esempio: campioni classe 1 nel test
    
    # Calcola elementi CM da recall e precision
    recall_0 = row['recall_0']
    precision_0 = row['precision_0']
    recall_1 = row['recall_1']
    
    # True Negatives (correttamente predetti come 0)
    tn = int(n_class_0 * recall_0)
    
    # False Positives (predetti come 0 ma sono 1)
    if precision_0 > 0:
        fp = int(tn / precision_0 - tn)
    else:
        fp = n_class_1 - int(n_class_1 * recall_1)
    
    # True Positives (correttamente predetti come 1)
    tp = int(n_class_1 * recall_1)
    
    # False Negatives (predetti come 1 ma sono 0)
    fn = n_class_0 - tn
    
    # Assicura che i totali siano corretti
    if tn + fn != n_class_0:
        fn = n_class_0 - tn
    if fp + tp != n_class_1:
        fp = n_class_1 - tp
    
    cm = np.array([[tn, fn], [fp, tp]])
    confusion_matrices[model_name] = cm
    
    print(f"  ✓ {model_name}")


📊 CREAZIONE DASHBOARD INTERATTIVA

📂 Caricamento dati...
✓ Caricati risultati per 5 modelli
📊 Generazione matrici di confusione...
  ✓ Random Undersampling + Ensemble
  ✓ NearMiss
  ✓ Tomek Links
  ✓ SMOTE + Undersampling
  ✓ BASELINE (Class Weighting)


In [17]:

# ==========================================
# FUNZIONE: CREA MATRICE DI CONFUSIONE
# ==========================================

def create_confusion_matrix_plot(cm, model_name, class_names=['Attrited', 'Existing']):
    """
    Crea un heatmap della confusion matrix con annotazioni TN/FP/FN/TP
    """
    tn, fn, fp, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    
    # Crea la matrice normalizzata per i colori
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Testo per ogni cella
    annotations = [
        [f"TN<br>{tn}<br>({cm_norm[0,0]*100:.1f}%)", 
         f"FN<br>{fn}<br>({cm_norm[0,1]*100:.1f}%)"],
        [f"FP<br>{fp}<br>({cm_norm[1,0]*100:.1f}%)", 
         f"TP<br>{tp}<br>({cm_norm[1,1]*100:.1f}%)"]
    ]
    
    # Colori personalizzati: diagonale (corretti) in verde, fuori diagonale in rosso
    colorscale = [
        [0, '#FFE5E5'],      # Rosso chiaro per errori bassi
        [0.5, '#FFA5A5'],    # Rosso medio
        [1, '#FF6B6B']       # Rosso scuro per errori alti
    ]
    
    # Per la diagonale usiamo verde
    z_colors = np.array([[0.9, 0.1], [0.1, 0.9]])  # Alta intensità sulla diagonale
    
    fig = go.Figure(data=go.Heatmap(
        z=cm,
        x=['Predicted<br>Attrited', 'Predicted<br>Existing'],
        y=['Actual<br>Attrited', 'Actual<br>Existing'],
        colorscale=[[0, '#E8F5E9'], [0.5, '#66BB6A'], [1, '#2E7D32']],
        text=annotations,
        texttemplate='%{text}',
        textfont={"size": 14, "color": "white", "family": "Arial Black"},
        hovertemplate='%{y} → %{x}<br>Count: %{z}<extra></extra>',
        showscale=False
    ))
    
    fig.update_layout(
        title=dict(
            text=f"<b>{model_name}</b>",
            x=0.5,
            xanchor='center',
            font=dict(size=16, color='#2C3E50')
        ),
        xaxis=dict(side='top', tickfont=dict(size=12)),
        yaxis=dict(tickfont=dict(size=12)),
        width=400,
        height=400,
        margin=dict(l=100, r=50, t=100, b=150),
        paper_bgcolor='white',
        plot_bgcolor='white'
    )
    
    # Aggiungi le metriche sotto la matrice
    metrics_text = (
        f"<b>True Negatives (TN):</b> {tn:,}  |  "
        f"<b>False Positives (FP):</b> {fp:,}<br>"
        f"<b>False Negatives (FN):</b> {fn:,}  |  "
        f"<b>True Positives (TP):</b> {tp:,}"
    )
    
    fig.add_annotation(
        text=metrics_text,
        xref="paper", yref="paper",
        x=0.5, y=-0.15,
        xanchor='center', yanchor='top',
        showarrow=False,
        font=dict(size=11, color='#34495E', family='Courier New'),
        bgcolor='#ECF0F1',
        bordercolor='#BDC3C7',
        borderwidth=1,
        borderpad=10
    )
    
    return fig

In [18]:
# ==========================================
# FUNZIONE: CREA GRAFICO RADAR
# ==========================================

def create_radar_chart(results_df, metrics, title, colors):
    """
    Crea un grafico radar per confrontare metriche multiple
    """
    fig = go.Figure()
    
    for idx, (_, row) in enumerate(results_df.iterrows()):
        values = [row[m] for m in metrics]
        values.append(values[0])  # Chiudi il poligono
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=metrics + [metrics[0]],
            fill='toself',
            name=row['model_name'],
            line=dict(width=2),
            opacity=0.6
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 1],
                tickfont=dict(size=10)
            ),
            angularaxis=dict(
                tickfont=dict(size=11, color='#2C3E50')
            )
        ),
        title=dict(
            text=f"<b>{title}</b>",
            x=0.5,
            xanchor='center',
            font=dict(size=14, color='#2C3E50')
        ),
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.3,
            xanchor="center",
            x=0.5,
            font=dict(size=10)
        ),
        height=450,
        paper_bgcolor='white',
        plot_bgcolor='white'
    )
    
    return fig


In [19]:
# ==========================================
# FUNZIONE: CREA BAR CHART COMPARATIVO
# ==========================================

def create_comparison_bars(results_df, metric, title, color, ylabel):
    """
    Crea un grafico a barre per confrontare una singola metrica
    """
    # Ordina per valore della metrica
    df_sorted = results_df.sort_values(metric, ascending=False)
    
    fig = go.Figure()
    
    # Colori: evidenzia il migliore in modo diverso
    colors = [color if i > 0 else '#27AE60' for i in range(len(df_sorted))]
    
    fig.add_trace(go.Bar(
        x=df_sorted['model_name'],
        y=df_sorted[metric],
        text=[f"{v:.4f}" for v in df_sorted[metric]],
        textposition='outside',
        marker=dict(
            color=colors,
            line=dict(color='#2C3E50', width=1.5)
        ),
        hovertemplate='<b>%{x}</b><br>' + ylabel + ': %{y:.4f}<extra></extra>'
    ))
    
    fig.update_layout(
        title=dict(
            text=f"<b>{title}</b>",
            x=0.5,
            xanchor='center',
            font=dict(size=14, color='#2C3E50')
        ),
        xaxis=dict(
            title="",
            tickangle=-45,
            tickfont=dict(size=10)
        ),
        yaxis=dict(
            title=ylabel,
            range=[0, max(df_sorted[metric]) * 1.15],
            tickfont=dict(size=11)
        ),
        height=400,
        showlegend=False,
        paper_bgcolor='white',
        plot_bgcolor='white',
        margin=dict(b=150)
    )
    
    # Aggiungi linea di riferimento per la media
    mean_val = df_sorted[metric].mean()
    fig.add_hline(
        y=mean_val, 
        line_dash="dash", 
        line_color="gray",
        annotation_text=f"Media: {mean_val:.4f}",
        annotation_position="right"
    )
    
    return fig

In [20]:
# ==========================================
# FUNZIONE: CREA HEATMAP METRICHE
# ==========================================

def create_metrics_heatmap(results_df, metrics, title):
    """
    Crea una heatmap con tutte le metriche per ogni modello
    """
    # Prepara i dati
    data_matrix = results_df[metrics].values
    model_names = results_df['model_name'].values
    
    # Nomi delle metriche più leggibili
    metric_labels = [m.replace('_', ' ').title() for m in metrics]
    
    fig = go.Figure(data=go.Heatmap(
        z=data_matrix,
        x=metric_labels,
        y=model_names,
        colorscale='RdYlGn',
        text=np.round(data_matrix, 4),
        texttemplate='%{text}',
        textfont={"size": 10},
        hovertemplate='<b>%{y}</b><br>%{x}: %{z:.4f}<extra></extra>',
        colorbar=dict(
            title="Score",
            tickfont=dict(size=10)
        )
    ))
    
    fig.update_layout(
        title=dict(
            text=f"<b>{title}</b>",
            x=0.5,
            xanchor='center',
            font=dict(size=14, color='#2C3E50')
        ),
        xaxis=dict(
            tickangle=-45,
            tickfont=dict(size=10),
            side='top'
        ),
        yaxis=dict(
            tickfont=dict(size=10)
        ),
        height=400,
        paper_bgcolor='white',
        plot_bgcolor='white'
    )
    
    return fig


In [21]:
# ==========================================
# CREAZIONE DASHBOARD COMPLETA
# ==========================================

print("\n🎨 Generazione visualizzazioni...")

# Lista per contenere tutte le figure
all_figures = []

# Header della dashboard
header_html = f"""
<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 10px; margin-bottom: 30px;'>
    <h1 style='margin: 0; font-size: 36px; font-weight: bold;'>📊 Dashboard Comparativa Approcci Resampling</h1>
    <p style='margin: 10px 0 0 0; font-size: 18px;'>Analisi Completa - Focus Classe Minoritaria (Churners)</p>
    <p style='margin: 5px 0 0 0; font-size: 14px; opacity: 0.9;'>🎯 Obiettivo: Massimizzare il Recall sulla Classe 0 (Attrited Customer)</p>
</div>
"""

# Sezione 1: Matrici di Confusione
print("  → Matrici di confusione...")
confusion_section = "<h2 style='color: #2C3E50; border-bottom: 3px solid #3498DB; padding-bottom: 10px;'>📋 1. Matrici di Confusione per Ogni Modello</h2>"

cm_figures = []
for model_name in results_df['model_name']:
    cm = confusion_matrices[model_name]
    fig = create_confusion_matrix_plot(cm, model_name)
    cm_figures.append(fig)

# Sezione 2: Metriche Classe Minoritaria (0 - Churners)
print("  → Metriche classe minoritaria...")
minority_section = "<h2 style='color: #2C3E50; border-bottom: 3px solid #E74C3C; padding-bottom: 10px; margin-top: 50px;'>⭐ 2. Metriche Classe Minoritaria (Attrited Customer - Classe 0)</h2>"
minority_section += "<p style='color: #7F8C8D; font-size: 14px;'>Focus principale: identificare correttamente i churner</p>"

# Radar chart classe minoritaria
fig_radar_0 = create_radar_chart(
    results_df,
    ['recall_0', 'precision_0', 'f1_0'],
    "Confronto Metriche - Classe 0 (Churners)",
    [COLOR_CLASS_0]
)

# Bar charts per ogni metrica classe 0
fig_recall_0 = create_comparison_bars(
    results_df, 'recall_0',
    "🎯 Recall Classe 0 (METRICA PRINCIPALE)",
    COLOR_CLASS_0, "Recall"
)

fig_precision_0 = create_comparison_bars(
    results_df, 'precision_0',
    "🎯 Precision Classe 0",
    COLOR_CLASS_0, "Precision"
)

fig_f1_0 = create_comparison_bars(
    results_df, 'f1_0',
    "🎯 F1-Score Classe 0",
    COLOR_CLASS_0, "F1-Score"
)

# Sezione 3: Metriche Classe Maggioritaria (1 - Existing)
print("  → Metriche classe maggioritaria...")
majority_section = "<h2 style='color: #2C3E50; border-bottom: 3px solid #1ABC9C; padding-bottom: 10px; margin-top: 50px;'>📊 3. Metriche Classe Maggioritaria (Existing Customer - Classe 1)</h2>"
majority_section += "<p style='color: #7F8C8D; font-size: 14px;'>Monitoraggio: assicurarsi di non degradare troppo questa classe</p>"

# Radar chart classe maggioritaria
fig_radar_1 = create_radar_chart(
    results_df,
    ['recall_1', 'precision_1', 'f1_1'],
    "Confronto Metriche - Classe 1 (Existing)",
    [COLOR_CLASS_1]
)

# Bar charts per ogni metrica classe 1
fig_recall_1 = create_comparison_bars(
    results_df, 'recall_1',
    "📊 Recall Classe 1",
    COLOR_CLASS_1, "Recall"
)

fig_precision_1 = create_comparison_bars(
    results_df, 'precision_1',
    "📊 Precision Classe 1",
    COLOR_CLASS_1, "Precision"
)

fig_f1_1 = create_comparison_bars(
    results_df, 'f1_1',
    "📊 F1-Score Classe 1",
    COLOR_CLASS_1, "F1-Score"
)

# Sezione 4: Metriche Globali
print("  → Metriche globali...")
global_section = "<h2 style='color: #2C3E50; border-bottom: 3px solid #9B59B6; padding-bottom: 10px; margin-top: 50px;'>🌍 4. Metriche Globali</h2>"
global_section += "<p style='color: #7F8C8D; font-size: 14px;'>Performance complessiva sui modelli</p>"

# Heatmap metriche globali
fig_heatmap_global = create_metrics_heatmap(
    results_df,
    ['balanced_acc', 'roc_auc', 'pr_auc'],
    "Heatmap Metriche Globali"
)

# Bar charts metriche globali
fig_balanced_acc = create_comparison_bars(
    results_df, 'balanced_acc',
    "⚖️ Balanced Accuracy",
    '#9B59B6', "Balanced Accuracy"
)

fig_roc_auc = create_comparison_bars(
    results_df, 'roc_auc',
    "📈 ROC-AUC",
    '#E67E22', "ROC-AUC"
)

fig_pr_auc = create_comparison_bars(
    results_df, 'pr_auc',
    "📈 PR-AUC",
    '#F39C12', "PR-AUC"
)

# Sezione 5: Score Aggregato Finale
print("  → Score aggregato...")
final_section = "<h2 style='color: #2C3E50; border-bottom: 3px solid #27AE60; padding-bottom: 10px; margin-top: 50px;'>🏆 5. Score Aggregato Finale</h2>"
final_section += "<p style='color: #7F8C8D; font-size: 14px;'>Formula: 60% Recall_0 + 20% Precision_0 + 10% F1_0 + 10% ROC-AUC</p>"

# Calcola score aggregato
results_df['weighted_score'] = (
    0.60 * results_df['recall_0'] +
    0.20 * results_df['precision_0'] +
    0.10 * results_df['f1_0'] +
    0.10 * results_df['roc_auc']
)

fig_final_score = create_comparison_bars(
    results_df, 'weighted_score',
    "🏆 Score Aggregato Pesato (Focus Recall Classe 0)",
    '#27AE60', "Score Aggregato"
)

# Crea heatmap riassuntiva di TUTTE le metriche
print("  → Heatmap riassuntiva completa...")
fig_heatmap_all = create_metrics_heatmap(
    results_df,
    ['recall_0', 'precision_0', 'f1_0', 'recall_1', 'precision_1', 'f1_1', 
     'balanced_acc', 'roc_auc', 'pr_auc', 'weighted_score'],
    "Heatmap Completa - Tutte le Metriche"
)



🎨 Generazione visualizzazioni...
  → Matrici di confusione...
  → Metriche classe minoritaria...
  → Metriche classe maggioritaria...
  → Metriche globali...
  → Score aggregato...
  → Heatmap riassuntiva completa...


In [22]:
# ==========================================
# SALVATAGGIO DASHBOARD HTML
# ==========================================

print("\n💾 Generazione file HTML...")

html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Dashboard Resampling Analysis</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            margin: 20px;
            background-color: #F8F9FA;
        }}
        .container {{
            max-width: 1400px;
            margin: 0 auto;
            background-color: white;
            padding: 30px;
            border-radius: 10px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        }}
        .grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(400px, 1fr));
            gap: 30px;
            margin: 30px 0;
        }}
        .full-width {{
            grid-column: 1 / -1;
        }}
        h2 {{
            margin-top: 50px;
        }}
        .info-box {{
            background: #E8F5E9;
            border-left: 5px solid #27AE60;
            padding: 15px;
            margin: 20px 0;
            border-radius: 5px;
        }}
        .warning-box {{
            background: #FFF3E0;
            border-left: 5px solid #F39C12;
            padding: 15px;
            margin: 20px 0;
            border-radius: 5px;
        }}
    </style>
</head>
<body>
    <div class="container">
        {header_html}
        
        <div class="info-box">
            <b>ℹ️ Informazioni Dashboard:</b><br>
            • <b>Classe 0 (Minoritaria - PRIORITARIA):</b> Attrited Customer (Churner) - Colore Rosso<br>
            • <b>Classe 1 (Maggioritaria):</b> Existing Customer - Colore Turchese<br>
            • <b>Obiettivo Principale:</b> Massimizzare il Recall della Classe 0 per catturare più churner possibili<br>
            • <b>Modelli Confrontati:</b> {len(results_df)} approcci diversi
        </div>
        
        {confusion_section}
        <div class="grid">
"""

# Aggiungi confusion matrices
for fig in cm_figures:
    html_content += f"<div>{fig.to_html(include_plotlyjs=False, div_id=f'cm_{cm_figures.index(fig)}')}</div>"

html_content += """
        </div>
        
        <div class="full-width">
"""

html_content += fig_heatmap_all.to_html(include_plotlyjs=False, div_id='heatmap_all')

html_content += f"""
        </div>
        
        {minority_section}
        <div class="warning-box">
            <b>⚠️ Focus Prioritario:</b> Il Recall della Classe 0 è la metrica più importante! 
            Vogliamo identificare il maggior numero possibile di churner, anche a costo di qualche falso positivo.
        </div>
        
        <div class="grid">
            <div class="full-width">{fig_radar_0.to_html(include_plotlyjs=False, div_id='radar_0')}</div>
            <div>{fig_recall_0.to_html(include_plotlyjs=False, div_id='recall_0')}</div>
            <div>{fig_precision_0.to_html(include_plotlyjs=False, div_id='precision_0')}</div>
            <div>{fig_f1_0.to_html(include_plotlyjs=False, div_id='f1_0')}</div>
        </div>
        
        {majority_section}
        <div class="info-box">
            <b>ℹ️ Nota:</b> È importante mantenere buone performance anche sulla classe maggioritaria 
            per evitare troppi falsi allarmi.
        </div>
        
        <div class="grid">
            <div class="full-width">{fig_radar_1.to_html(include_plotlyjs=False, div_id='radar_1')}</div>
            <div>{fig_recall_1.to_html(include_plotlyjs=False, div_id='recall_1')}</div>
            <div>{fig_precision_1.to_html(include_plotlyjs=False, div_id='precision_1')}</div>
            <div>{fig_f1_1.to_html(include_plotlyjs=False, div_id='f1_1')}</div>
        </div>
        
        {global_section}
        <div class="grid">
            <div class="full-width">{fig_heatmap_global.to_html(include_plotlyjs=False, div_id='heatmap_global')}</div>
            <div>{fig_balanced_acc.to_html(include_plotlyjs=False, div_id='balanced_acc')}</div>
            <div>{fig_roc_auc.to_html(include_plotlyjs=False, div_id='roc_auc')}</div>
            <div>{fig_pr_auc.to_html(include_plotlyjs=False, div_id='pr_auc')}</div>
        </div>
        
        {final_section}
        <div class="grid">
            <div class="full-width">{fig_final_score.to_html(include_plotlyjs=False, div_id='final_score')}</div>
        </div>
        
        <div class="info-box" style="margin-top: 50px;">
            <b>🎉 Conclusioni:</b><br>
            Il modello con lo score aggregato più alto è il migliore per il nostro caso d'uso.
            Ricorda che abbiamo dato priorità al Recall della classe minoritaria (60% del peso totale).
        </div>
        
        <div style="text-align: center; margin-top: 50px; padding: 20px; background: #ECF0F1; border-radius: 10px;">
            <p style="margin: 0; color: #7F8C8D;">Dashboard generata automaticamente • {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        </div>
    </div>
</body>
</html>
"""

# Salva il file HTML
output_file = 'dashboard_resampling_analysis.html'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"\n✅ Dashboard salvata: {output_file}")
print(f"   Apri il file nel browser per visualizzare la dashboard completa!")



💾 Generazione file HTML...

✅ Dashboard salvata: dashboard_resampling_analysis.html
   Apri il file nel browser per visualizzare la dashboard completa!


# Salvataggio del modello predittivo

In [15]:
# ==========================================
# SALVATAGGIO MODELLO MIGLIORE
# ==========================================
print("\n\n" + "="*80)
print("💾 SALVATAGGIO MODELLI E RISULTATI")
print("="*80)

# Determina quale modello salvare
if best_model['model_name'] == "BASELINE (Class Weighting)":
    best_pipeline = baseline_pipeline
    print("\n✓ Modello selezionato: BASELINE (Class Weighting)")
    print("  Pipeline completa (preprocessor + classifier)")
    
elif best_model['model_name'] == "Tomek Links":
    best_pipeline = {
        'preprocessor': preprocessor_for_resampling,
        'model': model_tomek,
        'resampler': tomek
    }
    print("\n✓ Modello selezionato: Tomek Links")
    print("  Componenti: preprocessor + resampler + classifier")
    
elif best_model['model_name'] == "NearMiss":
    best_pipeline = {
        'preprocessor': preprocessor_for_resampling,
        'model': model_nm,
        'resampler': nearmiss
    }
    print("\n✓ Modello selezionato: NearMiss")
    print("  Componenti: preprocessor + resampler + classifier")
    
elif best_model['model_name'] == "Random Undersampling + Ensemble":
    best_pipeline = {
        'preprocessor': preprocessor_for_resampling,
        'ensemble_models': ensemble_models
    }
    print("\n✓ Modello selezionato: Random Undersampling + Ensemble")
    print("  Componenti: preprocessor + ensemble di 5 modelli")
    
else:  # SMOTE + Undersampling
    best_pipeline = {
        'preprocessor': preprocessor_for_resampling,
        'model': model_hybrid,
        'smote': smote,
        'undersampler': rus
    }
    print("\n✓ Modello selezionato: SMOTE + Undersampling")
    print("  Componenti: preprocessor + SMOTE + undersampler + classifier")

# Salvataggio
MODEL_PATH = 'best_churn_model.pkl'
RESULTS_PATH = 'comparison_results.csv'
METADATA_PATH = 'model_metadata.pkl'

print(f"\nSalvataggio in corso...")

# Salva il modello
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(best_pipeline, f)
print(f"  ✓ Modello salvato: {MODEL_PATH}")

# Salva i risultati comparativi
results_df_sorted.to_csv(RESULTS_PATH, index=False)
print(f"  ✓ Risultati salvati: {RESULTS_PATH}")

# Salva metadata
metadata = {
    'label_encoder': le,
    'feature_names': list(X.columns),
    'best_model_name': best_model['model_name'],
    'best_score': best_model['weighted_score'],
    'training_date': time.strftime('%Y-%m-%d %H:%M:%S'),
    'class_distribution_train': dict(zip(*np.unique(y_train, return_counts=True))),
    'class_distribution_test': dict(zip(*np.unique(y_test, return_counts=True))),
    'best_metrics': {
        'recall_0': best_model['recall_0'],
        'precision_0': best_model['precision_0'],
        'f1_0': best_model['f1_0'],
        'recall_1': best_model['recall_1'],
        'precision_1': best_model['precision_1'],
        'f1_1': best_model['f1_1'],
        'balanced_acc': best_model['balanced_acc'],
        'roc_auc': best_model['roc_auc'],
        'pr_auc': best_model['pr_auc']
    }
}

with open(METADATA_PATH, 'wb') as f:
    pickle.dump(metadata, f)
print(f"  ✓ Metadata salvati: {METADATA_PATH}")



💾 SALVATAGGIO MODELLI E RISULTATI

✓ Modello selezionato: Random Undersampling + Ensemble
  Componenti: preprocessor + ensemble di 5 modelli

Salvataggio in corso...
  ✓ Modello salvato: best_churn_model.pkl
  ✓ Risultati salvati: comparison_results.csv
  ✓ Metadata salvati: model_metadata.pkl


# Utilizzo del modello salvato

In [ ]:
# ==========================================
# ESEMPIO DI UTILIZZO DEL MODELLO
# ==========================================
print("\n\n" + "="*80)
print("📖 GUIDA ALL'UTILIZZO DEL MODELLO SALVATO")
print("="*80)

print("\n# Caricamento del modello:")
print("```python")
print("import pickle")
print("import pandas as pd")
print("import numpy as np")
print("")
print("# Carica modello e metadata")
print("with open('best_churn_model.pkl', 'rb') as f:")
print("    model = pickle.load(f)")
print("")
print("with open('model_metadata.pkl', 'rb') as f:")
print("    metadata = pickle.load(f)")
print("")
print("# Prepara nuovi dati")
print("# new_data = pd.DataFrame(...)  # Stesso formato del training")
print("")

if best_model['model_name'] == "BASELINE (Class Weighting)":
    print("# Predizione (BASELINE):")
    print("predictions = model.predict(new_data)")
    print("probabilities = model.predict_proba(new_data)[:, 1]")
    
elif best_model['model_name'] == "Random Undersampling + Ensemble":
    print("# Predizione (ENSEMBLE):")
    print("X_new_processed = model['preprocessor'].transform(new_data)")
    print("ensemble_proba = np.zeros((len(X_new_processed), 2))")
    print("for m in model['ensemble_models']:")
    print("    ensemble_proba += m.predict_proba(X_new_processed)")
    print("ensemble_proba /= len(model['ensemble_models'])")
    print("predictions = np.argmax(ensemble_proba, axis=1)")
    print("probabilities = ensemble_proba[:, 1]")
    
else:
    print("# Predizione (CON RESAMPLING):")
    print("X_new_processed = model['preprocessor'].transform(new_data)")
    print("predictions = model['model'].predict(X_new_processed)")
    print("probabilities = model['model'].predict_proba(X_new_processed)[:, 1]")

print("")
print("# Decodifica delle predizioni")
print("le = metadata['label_encoder']")
print("predictions_labels = le.inverse_transform(predictions)")
print("```")


# Conclusioni 

In [14]:
# ==========================================
# RIEPILOGO FINALE
# ==========================================
print("\n\n" + "="*80)
print("✅ RIEPILOGO ESECUZIONE")
print("="*80)

print(f"\n📈 Approcci testati: {len(results_df)}")
print(f"🏆 Modello vincente: {best_model['model_name']}")
print(f"⭐ Score aggregato: {best_model['weighted_score']:.4f}")
print(f"🎯 Recall Classe 0 (Churners): {best_model['recall_0']:.4f}")

print("\n📁 File salvati:")
print(f"  - {MODEL_PATH} (modello completo)")
print(f"  - {RESULTS_PATH} (tabella comparativa)")
print(f"  - {METADATA_PATH} (configurazione e metriche)")

print("\n" + "="*80)
print("🎉 ANALISI COMPLETATA CON SUCCESSO!")
print("="*80)
print("\nProssimi passi consigliati:")
print("  1. Analizzare i falsi positivi/negativi del modello migliore")
print("  2. Considerare threshold tuning sulla probabilità")
print("  3. Validare su nuovi dati o con cross-validation")
print("  4. Integrare in pipeline di produzione")
print("  5. Monitorare performance nel tempo (model drift)")

print("\n" + "="*80)



✅ RIEPILOGO ESECUZIONE

📈 Approcci testati: 5
🏆 Modello vincente: Random Undersampling + Ensemble
⭐ Score aggregato: 0.7866
🎯 Recall Classe 0 (Churners): 0.8000

📁 File salvati:


NameError: name 'MODEL_PATH' is not defined